# M1 — Baseline pencarian lintas modal (SEA-VL Indonesia)
**IF25-40304 Pembelajaran Mesin Multimodal · Ganjil 2026/2027**

Notebook ini menjalankan pipeline end-to-end: membaca `train.csv` dan `val.csv`, menyematkan gambar dan caption dengan encoder pra-latih yang **dibekukan**, menghitung kesamaan kosinus, lalu mengembalikan peringkat. Hasilnya diukur dengan Recall@1/5/10 dan mAP@10 pada split validasi untuk empat arah pencarian.

In [1]:
import os, time, json, platform
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer

T0 = time.time()

# ---- konfigurasi ----
SEED      = 0
DATA_DIR  = "data"          # folder bundel LMS
IMG_DIR   = os.path.join(DATA_DIR, "images")
CACHE_DIR = "cache"         # embedding disimpan di sini agar M2 tidak mengulang
OUT_DIR   = "hasil_m1"
K_MAX     = 10
IMG_MODEL = "sentence-transformers/clip-ViTB-32"                # encoder gambar + teks Inggris
TXT_MODEL = "sentence-transformers/clip-ViTB-32-multilingual-v1" # encoder teks Indonesia
MODEL_TAG = IMG_MODEL.split("/")[-1] + "__" + TXT_MODEL.split("/")[-1]
BATCH_IMG, BATCH_TXT = 32, 64

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cpu"              # semua eksperimen harus berjalan tanpa GPU
torch.set_num_threads(os.cpu_count() or 1)
for d in (CACHE_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
print("torch", torch.__version__, "| threads", torch.get_num_threads(), "| device", DEVICE)

torch 2.14.1+cpu | threads 2 | device cpu


## 0. Impor bundel data dari zip

Upload zip bundel SEA-VL dari LMS ke lingkungan notebook (misalnya Colab atau Jupyter). Sel ini mengekstraknya, lalu mencari `train.csv`, `val.csv`, dan folder `images/` secara otomatis. Jika zip tidak terdeteksi, isi `ZIP_PATH` secara manual.

In [4]:
import os, glob, zipfile, shutil

EXTRACT_TO = "data"

# ---- pilih cara impor ----
# "upload" : di Google Colab, muncul dialog untuk memilih file zip dari komputer Anda
# "path"   : isi ZIP_PATH dengan lokasi zip (di Jupyter lokal atau sudah diunggah ke folder kerja)
IMPORT_MODE = "upload"
ZIP_PATH    = None     # contoh: "/home/nama/Downloads/sea-vl-id.zip"

def manual_import():
    if IMPORT_MODE == "upload":
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError("mode 'upload' hanya untuk Google Colab. Pakai IMPORT_MODE = 'path' dan isi ZIP_PATH.")
        print("Pilih file zip bundel SEA-VL dari komputer Anda...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("tidak ada file yang diupload")
        name = next(iter(uploaded))
        os.makedirs("uploads", exist_ok=True)
        zp = os.path.join("uploads", name)
        shutil.move(name, zp)
        return zp
    elif IMPORT_MODE == "path":
        if not ZIP_PATH or not os.path.exists(ZIP_PATH):
            raise FileNotFoundError(f"ZIP_PATH tidak ditemukan: {ZIP_PATH}")
        return ZIP_PATH
    else:
        raise ValueError("IMPORT_MODE harus 'upload' atau 'path'")

def locate_bundle(root):
    for dirpath, _, files in os.walk(root):
        if "train.csv" in files and "val.csv" in files:
            return dirpath
    return None

bundle = locate_bundle(EXTRACT_TO)
if bundle is None:
    zp = manual_import()
    print("mengekstrak:", zp)
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(zp) as z:
        z.extractall(EXTRACT_TO)
    bundle = locate_bundle(EXTRACT_TO)
assert bundle is not None, "train.csv / val.csv tidak ditemukan di dalam zip"

DATA_DIR = bundle
IMG_DIR  = os.path.join(DATA_DIR, "images") if os.path.isdir(os.path.join(DATA_DIR, "images")) else DATA_DIR
print("DATA_DIR :", DATA_DIR)
print("IMG_DIR  :", IMG_DIR)
print("ATTRIBUTION.md ada:", os.path.exists(os.path.join(DATA_DIR, "ATTRIBUTION.md")))
print("jumlah gambar:", len([f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]))

Pilih file zip bundel SEA-VL dari komputer Anda...


Saving bundle-mahasiswa-20261113T133902Z-1-001.zip to bundle-mahasiswa-20261113T133902Z-1-001.zip
mengekstrak: uploads/bundle-mahasiswa-20261113T133902Z-1-001.zip
DATA_DIR : data/bundle-mahasiswa
IMG_DIR  : data/bundle-mahasiswa/images
ATTRIBUTION.md ada: True
jumlah gambar: 2347


## 1. Muat data

In [5]:
def load_split(name):
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"))
    need = {"file", "caption_id", "caption_en"}
    missing = need - set(df.columns)
    assert not missing, f"kolom hilang di {name}.csv: {missing}"
    df = df.dropna(subset=list(need)).reset_index(drop=True)
    return df

def img_path(f):
    for p in (os.path.join(IMG_DIR, f), os.path.join(DATA_DIR, f), f):
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"gambar tidak ditemukan: {f}")

train_df = load_split("train")
val_df   = load_split("val")
print(len(train_df), "baris train |", len(val_df), "baris val")
val_df.head(3)

2072 baris train | 275 baris val


,image_id,file,caption_en,caption_id,native_lang,loc,category,relevance,caption_fit,img_quality,n_annotator,width,height,split
0,svl_140083990,images/svl_140083990.jpg,Nasi Padang using dendeng batokok and perkedel,Nasi padang pakai dendeng batokok dan perkedel...,Indonesian (ind),Indonesia,kuliner,5.0,1.0,1.0,2,955,1024,val
1,svl_140084024,images/svl_140084024.jpg,The picture is of balado potatoes. One of the ...,Gambar tersebut merupakan kentang balado. Sala...,Indonesian (ind),Indonesia,kuliner,5.0,1.0,1.0,2,576,1024,val
2,svl_140084031,images/svl_140084031.jpg,"This is ""sego goreng abang"" of East Java signa...",Iki sego goreng abang khas Jawa Timur,Javanese (jav),Indonesia,lain-lain,5.0,1.0,1.0,2,1024,768,val


## 2. Audit data (peran 1: Data dan Etika Data)

Memeriksa duplikat, sebaran kategori dan wilayah, serta kebocoran antara train dan val. Hasilnya dipakai untuk laporan kualitas data.

In [6]:
def audit(df, name):
    print(f"=== {name} ===")
    print("baris:", len(df), "| gambar unik:", df["file"].nunique(),
          "| caption_id unik:", df["caption_id"].nunique())
    print("baris dengan file duplikat:", int(df["file"].duplicated().sum()))
    print("baris dengan caption_en duplikat:", int(df["caption_en"].duplicated().sum()))
    if "category" in df:
        print("\nSebaran kategori:\n", df["category"].value_counts().to_string())
    if "loc" in df:
        print("\nSebaran wilayah (top 15):\n", df["loc"].value_counts().head(15).to_string())
    print()

audit(train_df, "train")
audit(val_df, "val")

norm = lambda s: s.astype(str).str.lower().str.strip()
shared_files = set(train_df.file) & set(val_df.file)
shared_caps  = set(norm(train_df.caption_en)) & set(norm(val_df.caption_en))
print("Gambar yang sama di train dan val :", len(shared_files))
print("caption_en yang sama di train dan val (lowercase):", len(shared_caps))

audit_summary = {
    "n_train": len(train_df), "n_val": len(val_df),
    "dup_file_train": int(train_df.file.duplicated().sum()),
    "dup_file_val": int(val_df.file.duplicated().sum()),
    "overlap_file_train_val": len(shared_files),
    "overlap_caption_train_val": len(shared_caps),
}
json.dump(audit_summary, open(os.path.join(OUT_DIR, "audit_summary.json"), "w"), indent=2)

=== train ===
baris: 2072 | gambar unik: 2072 | caption_id unik: 2025
baris dengan file duplikat: 0
baris dengan caption_en duplikat: 48

Sebaran kategori:
 category
kuliner                792
lain-lain              394
tarian                 243
arsitektur/bangunan    179
wayang/seni            132
tempat/alam            129
pakaian/tekstil         93
flora/fauna             58
transportasi            30
upacara/ritual          18
alat musik               4

Sebaran wilayah (top 15):
 loc
Indonesia                                                                                                      2010
Indonesia, Malaysia                                                                                              19
Indonesia, Malaysia, Singapore                                                                                   17
Indonesia, Thailand                                                                                               4
Indonesia, Singapore                     

## 3. Muat encoder (dibekukan)

Dimensi kedua encoder harus sama karena teks Indonesia disejajarkan ke ruang CLIP.

In [9]:
from sentence_transformers import SentenceTransformer

img_model = SentenceTransformer(IMG_MODEL, device=DEVICE)
txt_model = SentenceTransformer(TXT_MODEL, device=DEVICE)

d_img = img_model.get_sentence_embedding_dimension()
d_txt = txt_model.get_sentence_embedding_dimension()
print("dim gambar:", d_img, "| dim teks ID:", d_txt)
assert d_img == d_txt, "dimensi encoder tidak sama; penyejajaran tidak mungkin"

modules.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/1.91k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.03k [00:00<?, ?B/s]

0_CLIPModel/model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

0_CLIPModel/model.safetensors: downloading bytes:           |  0.00B            

... (20 outputs left out)

2_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 1.57MB            

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

dim gambar: 512 | dim teks ID: 512


/tmp/ipykernel_2347/4143705236.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  d_img = img_model.get_sentence_embedding_dimension()
/tmp/ipykernel_2347/4143705236.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  d_txt = txt_model.get_sentence_embedding_dimension()


## 4. Sematkan gambar dan caption (dengan cache)

In [10]:
def cached(name, fn):
    path = os.path.join(CACHE_DIR, f"{name}__{MODEL_TAG}.npy")
    if os.path.exists(path):
        print(f"cache dipakai: {name}")
        return np.load(path)
    t0 = time.time()
    arr = fn()
    np.save(path, arr)
    print(f"{name}: {arr.shape} dalam {time.time() - t0:.1f}s")
    return arr

def encode_images(files):
    out = []
    for i in range(0, len(files), BATCH_IMG):
        imgs = [Image.open(img_path(f)).convert("RGB") for f in files[i:i + BATCH_IMG]]
        out.append(img_model.encode(imgs, batch_size=BATCH_IMG,
                                    normalize_embeddings=True, convert_to_numpy=True))
        for im in imgs:
            im.close()
    return np.vstack(out)

def embed_split(name, df):
    E  = cached(f"{name}_img", lambda: encode_images(list(df.file)))
    Ti = cached(f"{name}_txtID", lambda: txt_model.encode(
        list(df.caption_id), batch_size=BATCH_TXT,
        normalize_embeddings=True, convert_to_numpy=True))
    Te = cached(f"{name}_txtEN", lambda: img_model.encode(
        list(df.caption_en), batch_size=BATCH_TXT,
        normalize_embeddings=True, convert_to_numpy=True))
    return E, Ti, Te

val_emb   = embed_split("val", val_df)
train_emb = embed_split("train", train_df)   # dipakai hanya untuk contoh query (bagian 6)

val_img: (275, 512) dalam 51.8s
val_txtID: (275, 512) dalam 18.5s
val_txtEN: (275, 512) dalam 14.8s
train_img: (2072, 512) dalam 407.4s
train_txtID: (2072, 512) dalam 110.7s
train_txtEN: (2072, 512) dalam 96.0s


## 5. Peringkat dan metrik

Relevansi didefinisikan dengan kunci `file`: sebuah teks relevan untuk gambar bila berasal dari gambar yang sama, dan sebaliknya. Ini menangani kasus satu gambar dengan lebih dari satu caption. Galeri gambar dideduplikasi agar satu gambar tidak dihitung dua kali.

- **Recall@k**: proporsi query yang punya setidaknya satu hasil relevan di k teratas.
- **AP@K**: rata-rata presisi pada posisi relevan di K teratas, dibagi min(jumlah relevan, K).
- **mAP@K**: rata-rata AP@K atas semua query.

In [11]:
def retrieval_metrics(Q, G, qkey, gkey, ks=(1, 5, 10), K=K_MAX):
    Qt, Gt = torch.from_numpy(np.ascontiguousarray(Q)), torch.from_numpy(np.ascontiguousarray(G))
    sims = Qt @ Gt.T                                          # kesamaan kosinus (ter-normalisasi)
    top = sims.topk(K, dim=1).indices.numpy()                 # peringkat teratas (nq, K)
    hits = (gkey[top] == qkey[:, None])                       # relevan? (nq, K)
    counts = pd.Series(gkey).value_counts()
    n_rel = np.array([counts.get(q, 0) for q in qkey])        # jumlah relevan per query
    valid = n_rel > 0
    out = {}
    for k in ks:
        out[f"R@{k}"] = float(hits[valid, :k].any(axis=1).mean())
    prec = np.cumsum(hits, axis=1) / np.arange(1, K + 1)
    ap = (prec * hits).sum(axis=1) / np.minimum(n_rel, K)
    out[f"mAP@{K}"] = float(ap[valid].mean())
    out["n_query"] = int(valid.sum())
    out["n_galeri"] = int(G.shape[0])
    out["acak_R@1"] = 1.0 / G.shape[0]
    return out

## 6. Empat arah pada split validasi

In [12]:
Ev, Tiv, Tev = val_emb
keys = val_df.file.values
uidx = val_df.index[~val_df.file.duplicated()].values     # indeks gambar unik
Eu, ku = Ev[uidx], keys[uidx]

directions = [
    ("gambar -> teks ID", Eu,  keys, Tiv, keys),
    ("gambar -> teks EN", Eu,  keys, Tev, keys),
    ("teks ID -> gambar", Tiv, keys, Eu,  ku),
    ("teks EN -> gambar", Tev, keys, Eu,  ku),
]
rows = [{"arah": name, **retrieval_metrics(Q, G, qk, gk)}
        for name, Q, qk, G, gk in directions]
metrics = pd.DataFrame(rows)
print(metrics.round(4).to_string(index=False))
metrics.to_csv(os.path.join(OUT_DIR, "metrics_val.csv"), index=False)

             arah    R@1    R@5   R@10  mAP@10  n_query  n_galeri  acak_R@1
gambar -> teks ID 0.2145 0.3745 0.5055  0.2930      275       275    0.0036
gambar -> teks EN 0.4400 0.7127 0.8327  0.5558      275       275    0.0036
teks ID -> gambar 0.1964 0.4182 0.5055  0.2830      275       275    0.0036
teks EN -> gambar 0.3818 0.6727 0.7709  0.5073      275       275    0.0036


## 7. Selisih kinerja bahasa Indonesia dan Inggris (language gap)

Selisih negatif berarti query bahasa Indonesia lebih lemah daripada bahasa Inggris pada gambar yang sama.

In [13]:
m = metrics.set_index("arah")
gap = pd.DataFrame({
    "gambar->teks (ID-EN)": m.loc["gambar -> teks ID"] - m.loc["gambar -> teks EN"],
    "teks->gambar (ID-EN)": m.loc["teks ID -> gambar"] - m.loc["teks EN -> gambar"],
}).loc[["R@1", "R@5", "R@10", f"mAP@{K_MAX}"]]
print(gap.round(4).to_string())
gap.to_csv(os.path.join(OUT_DIR, "language_gap_val.csv"))

        gambar->teks (ID-EN)  teks->gambar (ID-EN)
R@1                  -0.2255               -0.1855
R@5                  -0.3382               -0.2545
R@10                 -0.3273               -0.2655
mAP@10               -0.2628               -0.2243


## 8. Sepuluh contoh query Indonesia

Sesuai instruksi, contoh diambil dari `train.csv` agar tidak ada pengoptimalan pada val. Sepuluh baris dipilih secara acak dengan seed tetap dan **tidak dipilih berdasarkan hasilnya**. Galeri yang dipakai adalah gambar train. Contoh gagal ditampilkan apa adanya.

- **benar@1**: gambar peringkat 1 adalah gambar yang sesuai caption.
- **benar@3**: gambar yang sesuai muncul di 3 teratas.

In [14]:
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(train_df), size=10, replace=False)

Etr, Titr, _ = train_emb
tuidx = train_df.index[~train_df.file.duplicated()].values
Etr_u = Etr[tuidx]
tk = train_df.file.values[tuidx]

ex_rows = []
for qi in sample_idx:
    sims = Etr_u @ Titr[qi]
    top3 = np.argsort(-sims)[:3]
    preds = tk[top3]
    gold = train_df.file.iloc[qi]
    ex_rows.append({
        "query_ID": train_df.caption_id.iloc[qi],
        "caption_EN_asli": train_df.caption_en.iloc[qi],
        "gambar_benar": gold,
        "top1": preds[0], "top2": preds[1], "top3": preds[2],
        "benar@1": "ya" if preds[0] == gold else "tidak",
        "benar@3": "ya" if gold in preds else "tidak",
    })
examples = pd.DataFrame(ex_rows)
n_fail = int((examples["benar@1"] == "tidak").sum())
print(f"Contoh gagal (benar@1 = tidak): {n_fail} dari 10")
if n_fail < 2:
    print("PERHATIAN: kurang dari dua contoh gagal. Laporkan apa adanya, jangan memilih ulang sampel.")
examples.to_csv(os.path.join(OUT_DIR, "contoh_query_train.csv"), index=False)
examples

Contoh gagal (benar@1 = tidak): 9 dari 10


,query_ID,caption_EN_asli,gambar_benar,top1,top2,top3,benar@1,benar@3
0,"Mie Gacoan, mie pedas lengkap dengan Pangsit k...","Mie Gacoan, spicy noodles complete with dumpli...",images/svl_147562619.jpg,images/svl_147562460.jpg,images/svl_140084160.jpg,images/svl_147563025.jpg,tidak,tidak
1,Kerupuk rasa durian,Durian flavored crackers,images/svl_147525985.jpg,images/svl_144425111.jpg,images/svl_147563221.jpg,images/svl_144923843.jpg,tidak,tidak
2,Pagoda yang terletak di wilayah pecinan Pantai...,The pagoda located in the Chinatown area of ​​...,images/svl_144923838.jpg,images/svl_147562963.jpg,images/svl_140084271.jpg,images/svl_140084838.jpg,tidak,tidak
3,"Martabak manis, di dalamnya diolesi mentega, k...","Sweet martabak, smeared with butter, then spri...",images/svl_144425079.jpg,images/svl_147563017.jpg,images/svl_141004174.jpg,images/svl_141004052.jpg,tidak,tidak
4,Pemandangan dari jembatan penyebrangan orang d...,View from pedestrian bridge in front of Bandun...,images/svl_142113179.jpg,images/svl_144425477.jpg,images/svl_143497338.jpg,images/svl_143131948.jpg,tidak,tidak
5,Persiapan para penari yang akan tampil di karn...,The preparation of dancers getting ready to pe...,images/svl_140084412.jpg,images/svl_140084573.jpg,images/svl_141004067.jpg,images/svl_141004349.jpg,tidak,tidak
6,Becak di tengah kota Garut,Becak at the middle of Garut city,images/svl_140084172.jpg,images/svl_147525836.jpg,images/svl_140084984.jpg,images/svl_144425445.jpg,tidak,tidak
7,Para anak-anak yang berperan dalam tarian Wana...,The children playing the role of Wanara dancer...,images/svl_142113380.jpg,images/svl_140084771.jpg,images/svl_142113388.jpg,images/svl_140084418.jpg,tidak,tidak
8,Semangkok bakso dengan toge,a bowl of bakso with bean sprout,images/svl_141004216.jpg,images/svl_144425511.jpg,images/svl_142644617.jpg,images/svl_142113099.jpg,tidak,tidak
9,"Binatang khas penghuni Gunung Sindoro, yaitu b...",The characteristic animal inhabiting Mount Sin...,images/svl_140084591.jpg,images/svl_140084591.jpg,images/svl_140084597.jpg,images/svl_147562717.jpg,ya,ya


In [15]:
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(10, 4, figsize=(12, 30))
    for r, qi in enumerate(sample_idx):
        axes[r, 0].text(0.0, 0.5, train_df.caption_id.iloc[qi], fontsize=9, va="center", wrap=True)
        axes[r, 0].set_title(f"query {r+1}", fontsize=9)
        axes[r, 0].axis("off")
        sims = Etr_u @ Titr[qi]
        top3 = np.argsort(-sims)[:3]
        gold = train_df.file.iloc[qi]
        for c, j in enumerate(top3, start=1):
            ax = axes[r, c]
            ax.imshow(Image.open(img_path(tk[j])).convert("RGB"))
            ok = tk[j] == gold
            ax.set_title(("✓ " if ok else "✗ ") + f"peringkat {c}", fontsize=9, color="green" if ok else "red")
            ax.axis("off")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "contoh_query_train.png"), dpi=80)
    plt.show()
except ImportError:
    print("matplotlib tidak terpasang; grafik dilewati (tabel di atas tetap tersimpan).")

## 9. Catatan reproduksi

In [16]:
info = {
    "seed": SEED,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": DEVICE,
    "cpu_threads": torch.get_num_threads(),
    "img_model": IMG_MODEL,
    "txt_model": TXT_MODEL,
    "K": K_MAX,
    "n_train": len(train_df),
    "n_val": len(val_df),
    "waktu_total_detik": round(time.time() - T0, 1),
}
json.dump(info, open(os.path.join(OUT_DIR, "run_info.json"), "w"), indent=2)
print(json.dumps(info, indent=2))

{
  "seed": 0,
  "python": "3.13.18",
  "torch": "2.14.1+cpu",
  "device": "cpu",
  "cpu_threads": 2,
  "img_model": "sentence-transformers/clip-ViT-B-32",
  "txt_model": "sentence-transformers/clip-ViT-B-32-multilingual-v1",
  "K": 10,
  "n_train": 2072,
  "n_val": 275,
  "waktu_total_detik": 3533.9
}


---
**Atribusi data.** SEA-VL Indonesian subset (KORIKA-AI/sea-vl_crowdsourcing_id), diturunkan dari SEACrowd/sea-vl-crowdsourcing, lisensi **CC BY-SA 4.0** (https://creativecommons.org/licenses/by-sa/4.0/). Sitasi: Cahyawijaya, S., et al. (2025). *Crowdsource, Crawl, or Generate? Creating SEA-VL, a Multicultural Vision-Language Dataset for Southeast Asia*. ACL.